### All days of the challange:

* [Day 1: Handling missing values](https://www.kaggle.com/rtatman/data-cleaning-challenge-handling-missing-values)
* [Day 2: Scaling and normalization](https://www.kaggle.com/rtatman/data-cleaning-challenge-scale-and-normalize-data)
* [Day 3: Parsing dates](https://www.kaggle.com/rtatman/data-cleaning-challenge-parsing-dates/)
* [Day 4: Character encodings](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/)
* [Day 5: Inconsistent Data Entry](https://www.kaggle.com/rtatman/data-cleaning-challenge-inconsistent-data-entry/)
___
Welcome to day 2 of the 5-Day Data Challenge! Today, we're going to be looking at how to scale and normalize data (and what the difference is between the two!). To get started, click the blue "Fork Notebook" button in the upper, right hand corner. This will create a private copy of this notebook that you can edit and play with. Once you're finished with the exercises, you can choose to make your notebook public to share with others. :)

> **Your turn!** As we work through this notebook, you'll see some notebook cells (a block of either code or text) that has "Your Turn!" written in it. These are exercises for you to do to help cement your understanding of the concepts we're talking about. Once you've written the code to answer a specific question, you can run the code by clicking inside the cell (box with code in it) with the code you want to run and then hit CTRL + ENTER (CMD + ENTER on a Mac). You can also click in a cell and then click on the right "play" arrow to the left of the code. If you want to run all the code in your notebook, you can use the double, "fast forward" arrows at the bottom of the notebook editor.

Here's what we're going to do today:

* [Get our environment set up](#Get-our-environment-set-up)
* [Scaling vs. Normalization: What's the difference?](#Scaling-vs.-Normalization:-What's-the-difference?)
* [Practice scaling](#Practice-scaling)
* [Practice normalization](#Practice-normalization)

Let's get started!

# Get our environment set up
________

The first thing we'll need to do is load in the libraries and datasets we'll be using. 

> **Important!** Make sure you run this cell yourself or the rest of your code won't work!

In [ ]:
# modules we'll use
import os
import warnings
import pandas as pd
import numpy as np

# for Box-Cox Transformation
from scipy import stats

# for min_max scaling
try:
    from mlxtend.preprocessing import minmax_scaling
except ImportError:                       # si mlxtend no esta instalado, equivalente con scikit-learn
    from sklearn.preprocessing import MinMaxScaler

    def minmax_scaling(X, columns=None, min_val=0, max_val=1):
        X = pd.DataFrame(X)
        if columns is not None:
            X = X.iloc[:, columns]
        return pd.DataFrame(MinMaxScaler((min_val, max_val)).fit_transform(X.astype(float)),
                            columns=X.columns, index=X.index)

# plotting modules
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")   # sns.distplot esta deprecado y avisa en cada llamada

# read in all our data
# (la ruta original es la de Kaggle; aqui se busca tambien en las rutas locales habituales)
RUTAS = ["../input/kickstarter-projects/ks-projects-201801.csv",
         "input/kickstarter-projects/ks-projects-201801.csv",
         "ks-projects-201801.csv",
         "../ks-projects-201801.csv",
         os.path.expanduser("~/Downloads/ks-projects-201801.csv")]
RUTA = next((r for r in RUTAS if os.path.exists(r)), None)
if RUTA is None:
    raise FileNotFoundError(
        "No encuentro ks-projects-201801.csv. Descargalo de "
        "https://www.kaggle.com/datasets/kemical/kickstarter-projects "
        "y dejalo junto a este notebook.")

try:
    kickstarters_2017 = pd.read_csv(RUTA)
except UnicodeDecodeError:
    kickstarters_2017 = pd.read_csv(RUTA, encoding="ISO-8859-1")

print("Datos leidos de:", RUTA, "->", kickstarters_2017.shape)

# set seed for reproducibility
np.random.seed(0)
RANDOM_STATE = 0

Now that we're set up, let's learn about scaling & normalization. (If you like, you can take this opportunity to take a look at some of the data.)

# Scaling vs. Normalization: What's the difference?
____

One of the reasons that it's easy to get confused between scaling and normalization is because the terms are sometimes used interchangeably and, to make it even more confusing, they are very similar! In both cases, you're transforming the values of numeric variables so that the transformed data points have specific helpful properties. The difference is that, in scaling, you're changing the *range* of your data while in normalization you're changing the *shape of the distribution* of your data. Let's talk a little more in-depth about each of these options. 

___

## **Scaling**

This means that you're transforming your data so that it fits within a specific scale, like 0-100 or 0-1.  You want to scale data when you're using methods based on measures of how far apart data points, like [support vector machines, or SVM](https://en.wikipedia.org/wiki/Support_vector_machine) or [k-nearest neighbors, or KNN](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm). With these algorithms, a change of "1" in any numeric feature is given the same importance. 

For example, you might be looking at the prices of some products in both Yen and US Dollars. One US Dollar is worth about 100 Yen, but if you don't scale your prices methods like SVM or KNN will consider a difference in price of 1 Yen as important as a difference of 1 US Dollar! This clearly doesn't fit with our intuitions of the world. With currency, you can convert between currencies. But what about if you're looking at something like height and weight? It's not entirely clear how many pounds should equal one inch (or how many kilograms should equal one meter).

By scaling your variables, you can help compare different variables on equal footing. To help solidify what scaling looks like, let's look at a made-up example. (Don't worry, we'll work with real data in just a second, this is just to help illustrate my point.)


In [ ]:
# generate 1000 data points randomly drawn from an exponential distribution
original_data = np.random.exponential(size = 1000)

# mix-max scale the data between 0 and 1
scaled_data = minmax_scaling(original_data, columns = [0])

# plot both together to compare
fig, ax=plt.subplots(1,2)
sns.distplot(original_data, ax=ax[0])
ax[0].set_title("Original Data")
sns.distplot(scaled_data, ax=ax[1])
ax[1].set_title("Scaled data")

Notice that the *shape* of the data doesn't change, but that instead of ranging from 0 to 8ish, it now ranges from 0 to 1.

___
## Normalization

Scaling just changes the range of your data. Normalization is a more radical transformation. The point of normalization is to change your observations so that they can be described as a normal distribution.

> **[Normal distribution:](https://en.wikipedia.org/wiki/Normal_distribution)** Also known as the "bell curve", this is a specific statistical distribution where a roughly equal observations fall above and below the mean, the mean and the median are the same, and there are more observations closer to the mean. The normal distribution is also known as the Gaussian distribution.

In general, you'll only want to normalize your data if you're going to be using a machine learning or statistics technique that assumes your data is normally distributed. Some examples of these include t-tests, ANOVAs, linear regression, linear discriminant analysis (LDA) and Gaussian naive Bayes. (Pro tip: any method with "Gaussian" in the name probably assumes normality.)

The method were  using to normalize here is called the [Box-Cox Transformation](https://en.wikipedia.org/wiki/Power_transform#Box%E2%80%93Cox_transformation). Let's take a quick peek at what normalizing some data looks like:

In [ ]:
# normalize the exponential data with boxcox
normalized_data = stats.boxcox(original_data)

# plot both together to compare
fig, ax=plt.subplots(1,2)
sns.distplot(original_data, ax=ax[0])
ax[0].set_title("Original Data")
sns.distplot(normalized_data[0], ax=ax[1])
ax[1].set_title("Normalized data")

Notice that the *shape* of our data has changed. Before normalizing it was almost L-shaped. But after normalizing it looks more like the outline of a bell (hence "bell curve"). 

___
## Your turn!

For the following example, decide whether scaling or normalization makes more sense. 

* You want to build a linear regression model to predict someone's grades given how much time they spend on various activities during a normal school week.  You notice that your measurements for how much time students spend studying aren't normally distributed: some students spend almost no time studying and others study for four or more hours every day. Should you scale or normalize this variable?
* You're still working on your grades study, but you want to include information on how students perform on several fitness tests as well. You have information on how many jumping jacks and push-ups each student can complete in a minute. However, you notice that students perform far more jumping jacks than push-ups: the average for the former is 40, and for the latter only 10. Should you scale or normalize these variables?

# Practice scaling
___

To practice scaling and normalization, we're going to be using a dataset of Kickstarter campaigns. (Kickstarter is a website where people can ask people to invest in various projects and concept products.)

Let's start by scaling the goals of each campaign, which is how much money they were asking for.

In [ ]:
# select the usd_goal_real column
usd_goal = kickstarters_2017.usd_goal_real

# scale the goals from 0 to 1
scaled_data = minmax_scaling(usd_goal, columns = [0])

# plot the original & scaled data together to compare
fig, ax=plt.subplots(1,2)
sns.distplot(kickstarters_2017.usd_goal_real, ax=ax[0])
ax[0].set_title("Original Data")
sns.distplot(scaled_data, ax=ax[1])
ax[1].set_title("Scaled data")

You can see that scaling changed the scales of the plots dramatically (but not the shape of the data: it looks like most campaigns have small goals but a few have very large ones)

In [ ]:
# Your turn!

# We just scaled the "usd_goal_real" column. What about the "goal" column?

goal = kickstarters_2017.goal
scaled_goal = minmax_scaling(goal, columns=[0])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.distplot(goal, ax=ax[0])
ax[0].set_title("Original Data (goal)")
sns.distplot(scaled_goal, ax=ax[1])
ax[1].set_title("Scaled data (goal)")
plt.tight_layout()
plt.show()

print(f"goal          -> min {goal.min():>14,.0f}   max {goal.max():>16,.0f}")
print(f"goal escalado -> min {float(np.min(scaled_goal)):>14.6f}   max {float(np.max(scaled_goal)):>16.6f}")

In [ ]:
# El escalado funciona, pero 'goal' tiene un problema que 'usd_goal_real' no tiene:
# cada campana esta expresada en SU moneda, asi que la columna mezcla unidades distintas.
resumen_monedas = (kickstarters_2017
                   .groupby("currency")
                   .agg(campanas=("goal", "size"),
                        goal_mediana=("goal", "median"),
                        usd_goal_mediana=("usd_goal_real", "median"))
                   .sort_values("campanas", ascending=False))
print(resumen_monedas.round(0).to_string())

Esto es exactamente el ejemplo de los yenes y los dólares del principio del notebook, pero dentro de
una sola columna: el mínimo y el máximo que usa el escalado 0-1 salen de campañas en monedas distintas,
así que **el resultado no es comparable entre países**. `usd_goal_real` ya viene convertido a dólares,
y por eso es la columna correcta para escalar.

Regla práctica: **primero se unifican las unidades, después se escala.** Escalar no arregla un problema
de unidades, solo lo esconde detrás de números entre 0 y 1.

# Practice normalization
___

Ok, now let's try practicing normalization. We're going to normalize the amount of money pledged to each campaign.

In [ ]:
# get the index of all positive pledges (Box-Cox only takes postive values)
index_of_positive_pledges = kickstarters_2017.usd_pledged_real > 0

# get only positive pledges (using their indexes)
positive_pledges = kickstarters_2017.usd_pledged_real.loc[index_of_positive_pledges]

# normalize the pledges (w/ Box-Cox)
normalized_pledges = stats.boxcox(positive_pledges)[0]

# plot both together to compare
fig, ax=plt.subplots(1,2)
sns.distplot(positive_pledges, ax=ax[0])
ax[0].set_title("Original Data")
sns.distplot(normalized_pledges, ax=ax[1])
ax[1].set_title("Normalized data")

It's not perfect (it looks like a lot pledges got very few pledges) but it is much closer to normal!

In [ ]:
# Your turn!
# We looked at the usd_pledged_real column. What about the "pledged" column? Does it have the same info?

# Box-Cox solo admite valores estrictamente positivos, asi que de momento repetimos el filtro del tutorial
indice_pledged_positivo = kickstarters_2017.pledged > 0
pledged_positivo = kickstarters_2017.pledged.loc[indice_pledged_positivo]
pledged_normalizado = stats.boxcox(pledged_positivo)[0]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.distplot(pledged_positivo, ax=ax[0])
ax[0].set_title("Original Data (pledged)")
sns.distplot(pledged_normalizado, ax=ax[1])
ax[1].set_title("Normalized data (pledged)")
plt.tight_layout()
plt.show()

In [ ]:
# Does it have the same info?  -> no exactamente
par = kickstarters_2017.loc[indice_pledged_positivo, ["pledged", "usd_pledged_real", "currency"]]

print("Correlación de Pearson  :", round(par.pledged.corr(par.usd_pledged_real), 4))
print("Correlación de Spearman :", round(par.pledged.corr(par.usd_pledged_real, method="spearman"), 4))

difieren = (par.pledged - par.usd_pledged_real).abs() > 0.01 * par.pledged.clip(lower=1)
print(f"\nCampañas donde las dos columnas difieren más de un 1 %: {difieren.mean():.1%}")
print(f"De ellas, en USD: {(par.loc[difieren, 'currency'] == 'USD').mean():.1%}")

# el orden SI se conserva dentro de cada moneda, pero no entre monedas
print("\nSpearman calculada dentro de cada moneda:")
for moneda, g in par.groupby("currency"):
    if len(g) > 10:
        print(f"  {moneda:<5} {g.pledged.corr(g.usd_pledged_real, method='spearman'):>7.4f}   ({len(g):,} campañas)")

**No es la misma información.** `pledged` esta en la moneda de origen y `usd_pledged_real` en dólares:
dentro de una misma moneda el orden es idéntico (Spearman = 1), pero al mezclar monedas el ranking cambia,
porque 10 000 MXN y 10 000 GBP no valen lo mismo. La forma de la distribución es parecida
(ambas son fuertemente asimétricas a la derecha) y por eso Box-Cox las arregla igual de bien, pero para
cualquier comparación entre campañas hay que usar la columna en dólares.

Hay además una tercera columna, `usd pledged` (con espacio), que es la conversión *original* de Kaggle y
que **si tiene nulos**. Es la que usamos en la sección siguiente.

And that's it for today! If you have any questions, be sure to post them in the comments below or [on the forums](https://www.kaggle.com/questions-and-answers). 

Remember that your notebook is private by default, and in order to share it with other people or ask for help with it, you'll need to make it public. First, you'll need to save a version of your notebook that shows your current work by hitting the "Commit & Run" button. (Your work is saved automatically, but versioning your work lets you go back and look at what it was like at the point you saved it. It also lets you share a nice compiled notebook instead of just the raw code.) Then, once your notebook is finished running, you can go to the Settings tab in the panel to the left (you may have to expand it by hitting the [<] button next to the "Commit & Run" button) and setting the "Visibility" dropdown to "Public".

# More practice!
___

Try finding a new dataset and pretend you're preparing to preform a [regression analysis](https://www.kaggle.com/rtatman/the-5-day-regression-challenge). ([These datasets are a good start!](https://www.kaggle.com/rtatman/datasets-for-regression-analysis)) Pick three or four variables and decide if you need to normalize or scale any of them and, if you think you should, practice applying the correct technique.

___
# Parte 2 — Imputar en vez de dropear

El tutorial resuelve el problema de Box-Cox (que solo admite valores positivos) **descartando filas**:

```python
index_of_positive_pledges = kickstarters_2017.usd_pledged_real > 0
positive_pledges = kickstarters_2017.usd_pledged_real.loc[index_of_positive_pledges]
```

Es la salida rápida, y además es una salida silenciosa: el notebook nunca dice cuántas campañas se han
quedado fuera ni de qué tipo eran. En esta parte hacemos lo contrario:

1. Inventario de todos los huecos del dataset, incluidos los **nulos disfrazados** (fechas de 1970,
   países `N,0"`, estados `undefined`).
2. Medir **que cuesta dropear**: cuántas filas y, sobre todo, que sesgo introduce.
3. **Imputar**: comparar media, mediana, mediana por grupo, KNN, MICE e imputación basada en dominio,
   midiendo el error de cada una contra valores reales ocultos a propósito.
4. Normalizar **sin tirar nada**: Box-Cox desplazado y Yeo-Johnson en lugar del filtro `> 0`.

## 2.1 Inventario de huecos

In [ ]:
ks = kickstarters_2017.copy()

# fechas como fechas, y la duracion de cada campana (nos hara falta mas adelante)
ks["lanzamiento"] = pd.to_datetime(ks.launched, errors="coerce")
ks["cierre"] = pd.to_datetime(ks.deadline, errors="coerce")
ks["duracion"] = (ks.cierre - ks.lanzamiento).dt.days

print("Dimensiones:", ks.shape, "| duplicados:", ks.duplicated().sum(), "\n")

nulos = ks.isna().sum()
print("NULOS DECLARADOS (NaN)")
print(nulos[nulos > 0].to_string(), "\n")

print("NULOS DISFRAZADOS (no son NaN, pero son huecos)")
fecha_imposible = ks.lanzamiento.dt.year < 2009          # Kickstarter existe desde 2009
print(f"  launched anterior a 2009 (1970-01-01) : {fecha_imposible.sum():>7,}")
print(f"  country == 'N,0\"'                     : {(ks.country == 'N,0\"').sum():>7,}")
print(f"  state  == 'undefined'                 : {(ks.state == 'undefined').sum():>7,}")
print(f"  state  == 'live' (aún sin resultado)  : {(ks.state == 'live').sum():>7,}\n")

print("CEROS ESTRUCTURALES (no son huecos, pero Box-Cox los rechaza)")
print(f"  pledged == 0          : {(ks.pledged == 0).sum():>7,}  ({(ks.pledged == 0).mean():.1%})")
print(f"  usd_pledged_real <= 0 : {(ks.usd_pledged_real <= 0).sum():>7,}  ({(ks.usd_pledged_real <= 0).mean():.1%})")

Tres categorías muy distintas, y conviene no mezclarlas:

- **`usd pledged`** tiene nulos de verdad: la columna existe, el valor no se registró.
- **Las fechas de 1970** son un `0` de *timestamp* Unix que alguien guardó como fecha. Los mismos
  registros llevan `country = N,0"` y `state = undefined`: son filas rotas en origen.
- **Los ceros de `pledged`** no son huecos. Son información real y valiosísima: campañas que no
  recaudaron ni un dólar. Dropearlas para que Box-Cox funcione no es limpiar datos, es borrar el peor
  resultado posible del dataset.

## 2.2 Qué cuesta dropear

In [ ]:
sin_dato = ks["usd pledged"].isna()
no_positivo = ks.usd_pledged_real <= 0
descartables = sin_dato | no_positivo | fecha_imposible

print(f"Filas que el enfoque 'dropear' eliminaría: {descartables.sum():,} de {len(ks):,} "
      f"({descartables.mean():.1%})\n")

comparativa = pd.DataFrame({
    "dataset completo": ks.state.value_counts(normalize=True) * 100,
    "filas descartadas": ks.loc[descartables, "state"].value_counts(normalize=True) * 100,
    "filas supervivientes": ks.loc[~descartables, "state"].value_counts(normalize=True) * 100,
}).fillna(0).round(1)
print("Distribución del estado de la campaña (%)")
print(comparativa.to_string(), "\n")

exito_antes = (ks.state == "successful").mean()
exito_despues = (ks.loc[~descartables, "state"] == "successful").mean()
print(f"Tasa de éxito ANTES de dropear   : {exito_antes:.2%}")
print(f"Tasa de éxito DESPUÉS de dropear : {exito_despues:.2%}")
print(f"Sesgo introducido                : {(exito_despues - exito_antes) * 100:+.2f} puntos")

In [ ]:
# El sesgo tambien es por categoria: no todas pierden la misma proporcion de filas
perdida = (ks.assign(descartada=descartables)
             .groupby("main_category")
             .agg(campanas=("descartada", "size"), perdidas=("descartada", "sum")))
perdida["% perdido"] = (perdida.perdidas / perdida.campanas * 100).round(1)
perdida = perdida.sort_values("% perdido", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(perdida.index, perdida["% perdido"], color="#E45756")
ax.axvline(descartables.mean() * 100, color="black", ls="--", lw=1,
           label=f"media global ({descartables.mean():.1%})")
ax.invert_yaxis()
ax.set(xlabel="% de campañas que se perderían al dropear", title="Dropear no afecta por igual a todos")
ax.legend()
plt.tight_layout()
plt.show()

print(perdida[["campanas", "perdidas", "% perdido"]].to_string())

Aquí está el argumento de toda esta parte. Las filas que se van **no son una muestra aleatoria**: son
casi todas campañas fracasadas o canceladas. Dropear no reduce el dataset, lo **reescribe**: sube
artificialmente la tasa de éxito y castiga más a unas categorías que a otras.

Cualquier conclusión posterior ("las campañas de Kickstarter tienen un X % de éxito", "esta categoría
recauda más") sale contaminada, y el notebook no deja rastro de la decisión que la contaminó.

## 2.3 Imputar `usd pledged`: seis estrategias medidas, no elegidas a ojo

La forma honesta de elegir un imputador es **esconder datos que si tenemos**: tomamos filas con
`usd pledged` observado, ocultamos el 25 % al azar, imputamos y comparamos con el valor real.
Como la variable es muy asimétrica, el error se mide en escala logarítmica (MAE log) además del
error absoluto en dólares.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (activa IterativeImputer)
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler

VAR_IMP = ["goal", "usd_goal_real", "pledged", "usd_pledged_real", "backers", "duracion"]
OBJETIVO = "usd pledged"

# KNN y MICE son O(n^2) en memoria: se evaluan sobre una muestra
observadas = ks.loc[ks[OBJETIVO].notna() & ks[VAR_IMP].notna().all(axis=1)]
muestra = observadas.sample(n=min(6000, len(observadas)), random_state=RANDOM_STATE).copy()

rng = np.random.default_rng(RANDOM_STATE)
oculto = rng.random(len(muestra)) < 0.25          # nulos artificiales (MCAR)
verdad = muestra[OBJETIVO].to_numpy(float)

muestra_hueca = muestra.copy()
muestra_hueca.loc[oculto, OBJETIVO] = np.nan
print(f"Muestra: {len(muestra):,} filas | valores ocultados: {oculto.sum():,} ({oculto.mean():.0%})")


def medir(nombre, estimado):
    real, est = verdad[oculto], np.asarray(estimado, float)[oculto]
    return {"imputación": nombre,
            "MAE log": np.abs(np.log1p(np.clip(est, 0, None)) - np.log1p(real)).mean(),
            "MAE $": np.abs(est - real).mean(),
            "error mediano $": np.median(np.abs(est - real)),
            "R2": 1 - ((est - real) ** 2).sum() / ((real - real.mean()) ** 2).sum()}


resultados = []

# 1-2. media y mediana de la propia columna
for nombre, estrategia in [("media", "mean"), ("mediana", "median")]:
    est = SimpleImputer(strategy=estrategia).fit_transform(muestra_hueca[[OBJETIVO]]).ravel()
    resultados.append(medir(nombre, est))

# 3. mediana por categoría (imputacion condicionada al grupo)
por_grupo = muestra_hueca.groupby("main_category")[OBJETIVO].transform("median")
resultados.append(medir("mediana por categoría",
                        muestra_hueca[OBJETIVO].fillna(por_grupo).fillna(muestra_hueca[OBJETIVO].median())))

# 4-5. KNN y MICE sobre la matriz numerica (log1p + estandarizada para que las distancias tengan sentido)
matriz = np.log1p(np.clip(muestra_hueca[VAR_IMP + [OBJETIVO]].to_numpy(float), 0, None))
escalador = StandardScaler().fit(matriz)          # StandardScaler ignora los NaN
Z = escalador.transform(matriz)

for nombre, imputador in [("KNN (k=5)", KNNImputer(n_neighbors=5)),
                          ("MICE (IterativeImputer)", IterativeImputer(random_state=RANDOM_STATE,
                                                                       max_iter=10))]:
    est = np.expm1(escalador.inverse_transform(imputador.fit_transform(Z)))[:, -1]
    resultados.append(medir(nombre, est))

# 6. imputacion de dominio: 'usd pledged' es 'pledged' convertido a dolares
tipo_cambio = (ks.loc[ks.pledged > 0]
                 .assign(fx=lambda d: d.usd_pledged_real / d.pledged)
                 .groupby("currency").fx.median())
est_dominio = muestra_hueca[OBJETIVO].fillna(muestra_hueca.pledged * muestra_hueca.currency.map(tipo_cambio))
resultados.append(medir("dominio (pledged x tipo de cambio)", est_dominio))

tabla_imp = pd.DataFrame(resultados).set_index("imputación").sort_values("MAE log")
print()
print(tabla_imp.round(3).to_string())

In [ ]:
# Lo que de verdad importa: que le hace cada imputacion a la DISTRIBUCION
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
observado_log = np.log1p(verdad[oculto])

paneles = [("Valores reales (ocultados)", observado_log, "#54A24B"),
           ("Imputado con la mediana", np.log1p(np.full(oculto.sum(), muestra_hueca[OBJETIVO].median())), "#E45756"),
           ("Imputado con la regla de dominio", np.log1p(np.clip(est_dominio.to_numpy(float)[oculto], 0, None)), "#4C78A8")]
for ax, (titulo, datos, color) in zip(axes, paneles):
    ax.hist(datos, bins=40, color=color, alpha=.85)
    ax.set(title=titulo, xlabel="log1p(usd pledged)")
axes[0].set_ylabel("campanas")
plt.suptitle("La mediana no imputa una distribución: imputa un pico", y=1.03)
plt.tight_layout()
plt.show()

Dos lecciones:

1. **La imputación basada en dominio gana por goleada.** `usd pledged` no es un número cualquiera: es
   `pledged` convertido a dólares, así que conociendo esa relación el error se desploma frente a
   cualquier imputador estadístico (mira la columna `MAE log` de la tabla). MICE se le acerca porque
   acaba descubriendo sola esa relación lineal entre columnas; la mediana ni lo intenta. *Antes de
   llamar a un imputador hay que preguntarse si la columna se puede reconstruir a partir de otra.*
2. **Media y mediana destruyen la varianza.** El histograma central lo enseña: donde había una
   distribución, la mediana deja una barra. Con un 1 % de nulos eso apenas se nota; con un 20 %
   deformaría cualquier análisis posterior.

Aplicamos la ganadora al dataset completo, dejando además un **indicador de ausencia**: si el dato
faltaba por un motivo no aleatorio, el modelo podrá usarlo.

In [ ]:
ks["usd_pledged_faltante"] = ks["usd pledged"].isna().astype(int)
ks["usd pledged"] = ks["usd pledged"].fillna(ks.pledged * ks.currency.map(tipo_cambio))
ks["usd pledged"] = ks["usd pledged"].fillna(ks.usd_pledged_real)   # por si alguna moneda no tenia tipo

# fechas imposibles: en lugar de dropear las filas, reconstruimos el lanzamiento
# a partir del cierre y de la duracion tipica de su categoria
duracion_tipica = ks.loc[ks.duracion.between(1, 120)].groupby("main_category").duracion.median()
ks["fecha_imputada"] = fecha_imposible.astype(int)
reconstruido = ks.cierre - pd.to_timedelta(ks.main_category.map(duracion_tipica), unit="D")
ks.loc[fecha_imposible, "lanzamiento"] = reconstruido[fecha_imposible]
ks["duracion"] = (ks.cierre - ks.lanzamiento).dt.days

# nombres ausentes: placeholder + indicador (el largo del título será una variable del modelo)
ks["nombre_faltante"] = ks.name.isna().astype(int)
ks["name"] = ks.name.fillna("(sin titulo)")
ks["nombre_largo"] = ks.name.str.len()
ks["nombre_palabras"] = ks.name.str.split().str.len()

print("Nulos restantes:", int(ks.isna().sum().sum()))
print("Filas conservadas:", f"{len(ks):,} de {len(kickstarters_2017):,} (100 %)")
print("\nIndicadores creados:")
print(ks[["usd_pledged_faltante", "fecha_imputada", "nombre_faltante"]].sum().to_string())
print("\nDuración tras la imputación:")
print(ks.duracion.describe().round(1).to_string())

## 2.4 Normalizar sin tirar nada

Queda el motivo original del descarte: Box-Cox exige valores estrictamente positivos y hay
campañas con `usd_pledged_real == 0`. Tres alternativas al filtro:

| | Qué hace | Coste |
|---|---|---|
| **Filtrar `> 0`** (tutorial) | elimina las filas problemáticas | pierde datos y sesga la muestra |
| **Box-Cox desplazado** | transforma `x + 1`, que ya es positivo | conserva todo; el desplazamiento es arbitrario |
| **Yeo-Johnson** | variante de Box-Cox definida para ceros y negativos | conserva todo; no necesita truco |

In [ ]:
from sklearn.preprocessing import PowerTransformer

serie = ks.usd_pledged_real.to_numpy(float)

filtrado = stats.boxcox(serie[serie > 0])[0]                       # 1. dropear (tutorial)
desplazado, lam = stats.boxcox(serie + 1)                          # 2. Box-Cox desplazado
yeo = PowerTransformer(method="yeo-johnson").fit_transform(serie.reshape(-1, 1)).ravel()   # 3. Yeo-Johnson

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, (titulo, datos, color) in zip(axes, [
        ("Original", serie, "#BAB0AC"),
        (f"Box-Cox filtrando > 0\n({len(filtrado):,} filas)", filtrado, "#E45756"),
        (f"Box-Cox desplazado (+1)\n({len(desplazado):,} filas, lambda={lam:.3f})", desplazado, "#4C78A8"),
        (f"Yeo-Johnson\n({len(yeo):,} filas)", yeo, "#54A24B")]):
    ax.hist(datos, bins=60, color=color)
    ax.set_title(titulo, fontsize=10)
plt.suptitle("Tres formas de normalizar la misma columna", y=1.05)
plt.tight_layout()
plt.show()

resumen = pd.DataFrame({
    "filas usadas": [len(serie), len(filtrado), len(desplazado), len(yeo)],
    "% del dataset": [100, len(filtrado) / len(serie) * 100, 100, 100],
    "asimetría": [stats.skew(serie), stats.skew(filtrado), stats.skew(desplazado), stats.skew(yeo)],
    "curtosis": [stats.kurtosis(serie), stats.kurtosis(filtrado), stats.kurtosis(desplazado), stats.kurtosis(yeo)],
}, index=["original", "Box-Cox filtrando", "Box-Cox desplazado", "Yeo-Johnson"])
print(resumen.round(2).to_string())

El histograma del Box-Cox desplazado y el de Yeo-Johnson enseñan algo que el del tutorial **no puede
enseñar**: la barra gigante de la izquierda, las campañas que recaudaron exactamente 0. Esa es la
distribución real, bimodal — un pico en cero y una campaña a la derecha — y el filtro `> 0` la
convertía en una distribución unimodal inventada.

Compara la asimetría de la tabla: la versión filtrada es la que mejor puntúa, y no porque sea mejor
transformación, sino porque **le hemos quitado la parte difícil de los datos**. Conservar los ceros
cuesta normalidad; el resultado con todos los datos es más feo y más cierto. Si lo que se busca es normalidad para un modelo
lineal, la solución honesta no es esconder los ceros sino modelarlos aparte (un modelo para *recauda o
no recauda* y otro para *cuánto*), o usar un método que no exija normalidad.

___
# Parte 3 — Balanceo de clases

Ya tenemos el dataset **completo** (378 661 campañas, cero filas descartadas). Ahora el otro problema
clásico de la limpieza de datos: cuando la variable que queremos predecir tiene clases muy desiguales,
un modelo puede sacar un 99 % de acierto sin haber aprendido nada.

El dataset ofrece los dos escenarios en la misma columna `state`:

| Objetivo | Pregunta | Desbalanceo |
|---|---|---|
| `successful` | ¿la campaña alcanzará su meta? | **moderado**, en torno a 1:1,5 |
| `suspended` | ¿la campaña será suspendida por Kickstarter? | **severo**, en torno a 1:200 |

## 3.1 Definir el objetivo sin fuga de información

Antes de balancear nada hay que elegir las variables, y aquí hay una trampa fácil de pisar:
`pledged`, `backers`, `usd_pledged_real` y `usd pledged` **son el resultado de la campaña**, no datos
disponibles antes de lanzarla. Un modelo que las use acertará el 99,9 % y no servirá para nada, porque
en el momento en el que alguien querría usarlo (antes de lanzar) esas columnas todavía no existen.

Nos quedamos solo con información **pre-lanzamiento**: meta, duración, categoría, país, moneda, fecha
y título.

In [ ]:
finalizadas = ks[ks.state.isin(["successful", "failed", "canceled", "suspended"])].copy()
finalizadas["mes"] = finalizadas.lanzamiento.dt.month
finalizadas["anyo"] = finalizadas.lanzamiento.dt.year
finalizadas["log_meta"] = np.log1p(finalizadas.usd_goal_real)

VAR_NUM = ["log_meta", "duracion", "nombre_largo", "nombre_palabras", "mes", "anyo"]
VAR_CAT = ["main_category", "category", "country", "currency"]
VAR_FLAG = ["nombre_faltante"]

# excluidas a proposito: resultado de la campana (fuga), y el indicador usd_pledged_faltante,
# que tambien es informacion posterior al lanzamiento
FUGA = ["pledged", "backers", "usd_pledged_real", "usd pledged", "usd_pledged_faltante", "state"]

print("Variables del modelo :", VAR_NUM + VAR_CAT + VAR_FLAG)
print("Excluidas por fuga   :", FUGA, "\n")

reparto = finalizadas.state.value_counts()
print(reparto.to_string())
print()
for nombre, y in [("éxito (successful)", (finalizadas.state == "successful")),
                  ("suspensión (suspended)", (finalizadas.state == "suspended"))]:
    print(f"{nombre:<24} positivos: {y.sum():>7,}  ({y.mean():6.2%})  ->  1:{(1 - y.mean()) / y.mean():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (titulo, y) in zip(axes, [("Objetivo moderado: successful", (finalizadas.state == "successful")),
                                  ("Objetivo severo: suspended", (finalizadas.state == "suspended"))]):
    conteo = y.value_counts().sort_index()
    ax.bar(["negativa", "positiva"], conteo.values, color=["#4C78A8", "#E45756"])
    for i, v in enumerate(conteo.values):
        ax.text(i, v, f"{v:,}\n({v / len(y):.2%})", ha="center", va="bottom", fontsize=9)
    ax.set(title=titulo, ylim=(0, len(y) * 1.15))
plt.tight_layout()
plt.show()

## 3.2 El caso severo: predecir campañas suspendidas

Una suspensión es Kickstarter retirando una campaña (normalmente por incumplir las reglas o por
sospecha de fraude). Es el caso de uso típico de detección: **la clase que interesa es la rara**.

Metodología, en el orden correcto:

1. Partir train/test **estratificado** y dejar el test intacto, con la prevalencia real.
2. Codificar (escalar numéricas, one-hot categóricas) ajustando **solo con el train**.
3. Balancear **solo el train**. Nunca el test: si se remuestrea el test, las métricas describen un
   mundo que no existe.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

X = finalizadas[VAR_NUM + VAR_CAT + VAR_FLAG]
y = (finalizadas.state == "suspended").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

# tope de tamano para que el notebook corra en minutos y no en horas (subelo si quieres el 100 %)
MAX_TRAIN = 150_000
if len(X_train) > MAX_TRAIN:
    X_train, _, y_train, _ = train_test_split(X_train, y_train, train_size=MAX_TRAIN,
                                              stratify=y_train, random_state=RANDOM_STATE)

preprocesador = ColumnTransformer([
    ("num", StandardScaler(), VAR_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=200), VAR_CAT),
    ("flags", "passthrough", VAR_FLAG),
])
X_train_cod = preprocesador.fit_transform(X_train)
X_test_cod = preprocesador.transform(X_test)

print(f"train : {X_train_cod.shape[0]:>7,} filas | positivos {y_train.sum():>5,} ({y_train.mean():.2%})")
print(f"test  : {X_test_cod.shape[0]:>7,} filas | positivos {y_test.sum():>5,} ({y_test.mean():.2%})")
print(f"variables tras codificar: {X_train_cod.shape[1]}")

In [ ]:
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as PipelineImb

TECNICAS = {
    "RandomOverSampler":      RandomOverSampler(random_state=RANDOM_STATE),
    "SMOTE":                  SMOTE(random_state=RANDOM_STATE),
    "SMOTE moderado (1:10)":  SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE),
    "RandomUnderSampler":     RandomUnderSampler(random_state=RANDOM_STATE),
    "Under moderado (1:10)":  RandomUnderSampler(sampling_strategy=0.1, random_state=RANDOM_STATE),
}

filas = [{"técnica": "- sin balanceo -", "clase 0": int((y_train == 0).sum()),
          "clase 1": int((y_train == 1).sum()), "total": len(y_train),
          "% minoritaria": y_train.mean() * 100}]
for nombre, tecnica in TECNICAS.items():
    _, yr = tecnica.fit_resample(X_train_cod, y_train)
    filas.append({"técnica": nombre, "clase 0": int((yr == 0).sum()), "clase 1": int((yr == 1).sum()),
                  "total": len(yr), "% minoritaria": yr.mean() * 100})

tabla_remuestreo = pd.DataFrame(filas).set_index("técnica").round(1)
tabla_remuestreo["delta filas"] = tabla_remuestreo["total"] - len(y_train)
print(tabla_remuestreo.to_string())

La tabla resume las dos familias: el sobremuestreo **inventa** minoritarios hasta igualar (el train casi
se dobla) y el submuestreo **tira** mayoritarios hasta igualar (se queda con el 1 % de los datos).
Las variantes moderadas buscan un 1:10 en lugar de 1:1, que suele ser suficiente y mucho más barato.

### Evaluación sobre el test intacto

In [ ]:
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             balanced_accuracy_score, roc_auc_score, average_precision_score,
                             confusion_matrix, precision_recall_curve)

def evaluar(sampler, nombre, **kw_clf):
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **kw_clf)
    pasos = ([("balanceo", sampler)] if sampler is not None else []) + [("clf", clf)]
    modelo = PipelineImb(pasos).fit(X_train_cod, y_train)
    proba = modelo.predict_proba(X_test_cod)[:, 1]
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return proba, {"estrategia": nombre,
                   "accuracy": accuracy_score(y_test, pred),
                   "recall": recall_score(y_test, pred),
                   "precisión": precision_score(y_test, pred, zero_division=0),
                   "F1": f1_score(y_test, pred),
                   "bal. acc.": balanced_accuracy_score(y_test, pred),
                   "ROC-AUC": roc_auc_score(y_test, proba),
                   "PR-AUC": average_precision_score(y_test, proba),
                   "TP": tp, "FN": fn, "FP": fp}

probas, filas = {}, []
configuraciones = [(None, "- sin balanceo -", {}),
                   (None, "class_weight='balanced'", {"class_weight": "balanced"})]
configuraciones += [(t, n, {}) for n, t in TECNICAS.items()]

for sampler, nombre, kw in configuraciones:
    p, fila = evaluar(sampler, nombre, **kw)
    probas[nombre] = p
    filas.append(fila)
    print(f"  {nombre:<26} recall {fila['recall']:.1%}  precisión {fila['precisión']:.1%}")

tabla = pd.DataFrame(filas).set_index("estrategia")
print()
print(tabla[["accuracy", "recall", "precisión", "F1", "bal. acc.", "ROC-AUC", "PR-AUC",
             "TP", "FN", "FP"]].round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

m = tabla[["recall", "precisión", "F1"]]
pos = np.arange(len(m))
for i, (col, color) in enumerate(zip(m.columns, ["#E45756", "#4C78A8", "#54A24B"])):
    axes[0].barh(pos + (1 - i) * 0.27, m[col], 0.27, label=col, color=color)
axes[0].set(yticks=pos, yticklabels=m.index, xlim=(0, 1.05),
            title="Sobre el test intacto (umbral 0,5)")
axes[0].legend(loc="lower right")
axes[0].invert_yaxis()

coste = tabla.sort_values("FP")
pos2 = np.arange(len(coste))
axes[1].barh(pos2, coste["FP"], color="#BAB0AC", label="falsas alarmas (FP)")
axes[1].barh(pos2, coste["TP"], color="#E45756", label="suspensiones detectadas (TP)")
for j, (fp, tp) in enumerate(zip(coste["FP"], coste["TP"])):
    axes[1].text(fp + max(coste["FP"]) * .02, j, f"{tp} de {int(y_test.sum())}  ·  {fp} FP", va="center", fontsize=8)
axes[1].set(yticks=pos2, yticklabels=coste.index, xlabel=f"campañas del test ({len(y_test):,} en total)",
            title="El precio de detectar", xlim=(0, max(coste["FP"].max(), 1) * 1.45))
axes[1].legend(loc="lower right", bbox_to_anchor=(1, -0.30), ncol=2, frameon=False)
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión: el antes y el despues
seleccion = ["- sin balanceo -", "class_weight='balanced'", "SMOTE", "RandomUnderSampler"]
fig, axes = plt.subplots(1, len(seleccion), figsize=(17, 3.8))
for ax, nombre in zip(axes, seleccion):
    cm = confusion_matrix(y_test, (probas[nombre] >= 0.5).astype(int))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=12,
                    color="white" if cm[i, j] > cm.max() * .5 else "black")
    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=["pred. normal", "pred. suspendida"],
           yticklabels=["real normal", "real suspendida"],
           title=f"{nombre}\nrecall = {tabla.loc[nombre, 'recall']:.0%}")
plt.tight_layout()
plt.show()

In [ ]:
# ¿Mejora el balanceo la capacidad de ORDENAR por riesgo, o solo mueve el umbral?
fig, ax = plt.subplots(figsize=(8, 5))
for nombre in ["- sin balanceo -", "class_weight='balanced'", "SMOTE", "RandomUnderSampler"]:
    pr, rc, _ = precision_recall_curve(y_test, probas[nombre])
    ax.plot(rc, pr, lw=1.6, label=f"{nombre} (AP = {tabla.loc[nombre, 'PR-AUC']:.3f})")
ax.axhline(y_test.mean(), color="k", ls=":", lw=1, label=f"azar ({y_test.mean():.4f})")
ax.set(xlabel="recall", ylabel="precisión", title="Curva Precisión-Recall (clase suspendida)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("ROC-AUC por estrategia:")
print(tabla["ROC-AUC"].round(4).to_string())

In [ ]:
# La alternativa barata: no tocar los datos, solo bajar el umbral del modelo sin balancear
proba_base = probas["- sin balanceo -"]
filas_umbral = []
for u in [0.50, 0.10, 0.05, 0.02, 0.01, 0.007, 0.005, 0.003]:
    pred = (proba_base >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    filas_umbral.append({"umbral": u, "recall": recall_score(y_test, pred),
                         "precisión": precision_score(y_test, pred, zero_division=0),
                         "F1": f1_score(y_test, pred), "TP": tp, "FN": fn, "FP": fp})

barrido = pd.DataFrame(filas_umbral).set_index("umbral").round(4)
mejor = barrido["F1"].idxmax()
print(barrido.to_string())
print(f"\nMejor F1 moviendo sólo el umbral : {barrido.loc[mejor, 'F1']:.4f}  (umbral = {mejor})")
print(f"Mejor F1 con remuestreo          : {tabla['F1'].max():.4f}  ({tabla['F1'].idxmax()})")

## 3.3 El caso moderado: predecir el éxito

Para comparar, el mismo experimento con un desbalanceo suave (1:1,5). Aquí la pregunta es si el
balanceo sigue haciendo falta.

In [ ]:
exito = finalizadas[finalizadas.state.isin(["successful", "failed"])]
Xe = exito[VAR_NUM + VAR_CAT + VAR_FLAG]
ye = (exito.state == "successful").astype(int)

Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(Xe, ye, test_size=0.25, stratify=ye,
                                              random_state=RANDOM_STATE)
if len(Xe_tr) > MAX_TRAIN:
    Xe_tr, _, ye_tr, _ = train_test_split(Xe_tr, ye_tr, train_size=MAX_TRAIN, stratify=ye_tr,
                                          random_state=RANDOM_STATE)

pre_e = ColumnTransformer([("num", StandardScaler(), VAR_NUM),
                           ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=200), VAR_CAT),
                           ("flags", "passthrough", VAR_FLAG)])
Xe_tr_cod, Xe_te_cod = pre_e.fit_transform(Xe_tr), pre_e.transform(Xe_te)
print(f"train {Xe_tr_cod.shape} | positivos {ye_tr.mean():.1%}")

filas_e = []
for sampler, nombre, kw in [(None, "- sin balanceo -", {}),
                            (None, "class_weight='balanced'", {"class_weight": "balanced"}),
                            (RandomOverSampler(random_state=RANDOM_STATE), "RandomOverSampler", {}),
                            (SMOTE(random_state=RANDOM_STATE), "SMOTE", {}),
                            (RandomUnderSampler(random_state=RANDOM_STATE), "RandomUnderSampler", {})]:
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **kw)
    pasos = ([("balanceo", sampler)] if sampler is not None else []) + [("clf", clf)]
    modelo = PipelineImb(pasos).fit(Xe_tr_cod, ye_tr)
    proba = modelo.predict_proba(Xe_te_cod)[:, 1]
    pred = (proba >= 0.5).astype(int)
    filas_e.append({"estrategia": nombre, "accuracy": accuracy_score(ye_te, pred),
                    "recall": recall_score(ye_te, pred),
                    "precisión": precision_score(ye_te, pred, zero_division=0),
                    "F1": f1_score(ye_te, pred), "bal. acc.": balanced_accuracy_score(ye_te, pred),
                    "ROC-AUC": roc_auc_score(ye_te, proba)})

tabla_exito = pd.DataFrame(filas_e).set_index("estrategia")
print()
print(tabla_exito.round(4).to_string())

___
# Conclusiones

### Imputar en vez de dropear

| | |
|---|---|
| Filas que el filtro del tutorial elimina | las de la sección 2.2 (campañas con 0 recaudado, nulos y fechas rotas) |
| Qué tenían en común | casi todas `failed` o `canceled` |
| Efecto | la tasa de éxito del dataset sube artificialmente |
| Mejor imputador medido | la **regla de dominio** (`pledged` x tipo de cambio), no la mediana ni KNN |
| Filas conservadas al final | **100 %** |

1. **Dropear nunca es neutral.** Si las filas incompletas se parecen entre si —y aquí se parecían
   muchísimo: eran las campañas fracasadas— eliminarlas no reduce el dataset, lo reescribe.
2. **La mejor imputación suele ser la que usa conocimiento del dominio.** Antes de llamar a KNN o a
   MICE conviene mirar si la columna se puede reconstruir a partir de otra.
3. **Media y mediana matan la varianza.** Sustituyen una distribución por un pico; con pocos nulos da
   igual, con muchos deforma todo el análisis.
4. **Los huecos no siempre son NaN.** Fechas de 1970, países `N,0"` y estados `undefined` son nulos
   disfrazados: `isnull()` no los ve.
5. **Guardar un indicador de ausencia es gratis.** Si el dato falta por un motivo no aleatorio, esa
   bandera es información; si falta al azar, el modelo la ignorará.

### Balanceo de clases

| | |
|---|---|
| Objetivo severo | `suspended`, en torno al 0,5 % de positivos (1:200) |
| Sin balancear | accuracy altisima con **recall prácticamente 0**: no detecta nada |
| Con balanceo | el recall sube de golpe, la precisión se hunde |
| ROC-AUC / PR-AUC | apenas se mueven: compara las curvas antes de celebrar el recall |

1. **La exactitud miente con clases desbalanceadas.** El 99,5 % del modelo base corresponde a un
   clasificador que no detecta ni una suspensión. Recall, F1, bal. acc. y PR-AUC son las métricas
   honestas.
2. **El remuestreo es en gran medida un ajuste encubierto del umbral.** Si las curvas PR de la sección
   3.2 salen superpuestas -que es lo habitual con modelos lineales-, el balanceo no está ordenando
   mejor las campañas por riesgo: solo recalibra las probabilidades para que el corte de 0,5 empiece a
   marcar a alguien. El barrido de umbral lo comprueba, y consigue lo mismo sin tocar los datos.
3. **Empezar siempre por `class_weight='balanced'`.** Es una linea de código, no inventa datos, no tira
   datos y suele igualar al remuestreo.
4. **Con desbalanceo moderado (1:1,5) el balanceo cambia mucho menos.** En la sección 3.3 el ROC-AUC
   -que mide la ordenación, no el corte- se mueve en las últimas cifras; lo único que cambia de verdad
   es donde cae el umbral. El remuestreo es una herramienta para los casos extremos.
5. **El test no se toca nunca.** Ni se imputa con sus estadísticos ni se remuestrea: debe mantener la
   prevalencia real para que las métricas describan lo que pasaría en producción.

### Y las dos partes juntas

El detalle que une todo el notebook: **las filas que el tutorial descartaba eran justo las de la clase
minoritaria**. Dropear primero y balancear después es tirar ejemplos raros para luego pagar por
fabricar ejemplos raros sintéticos. El orden correcto es el contrario: conservar todo lo que se pueda
imputar, y solo entonces decidir si el desbalanceo que queda necesita tratamiento.